# D405 — Column Masking and Row Access in Snowflake

This lab extends the supplied customer-email example with Aadhaar, Indian tax PAN, bank account, phone, birth date, region, and balance fields. Every record is synthetic. Identifier strings are deliberately invalid test placeholders, not real identity or banking information. 

## 1. Two different questions: what can I see, and whose records can I see?

Imagine a bank employee looking at a customer register.

- **Column masking:** The customer row appears, but an email or account number is hidden or partially shown.
- **Row filtering:** Customers outside the employee's approved region do not appear at all.

People sometimes call the second operation “row masking.” Snowflake calls the enforcing object a **row access policy**. It filters records rather than replacing each row with asterisks.

| Control | What changes in the result? | Example |
|---|---|---|
| Column masking | Selected field values | Email becomes `********` |
| Row access policy | Which records appear | South analyst sees South customers only |
| Both together | Records and their visible values | Only South customers, with concealed identifiers |

The original records remain stored. Neither control deletes or anonymizes the source data. [Dynamic Data Masking](https://docs.snowflake.com/en/user-guide/security-column-ddm-intro); [row access policies](https://docs.snowflake.com/en/user-guide/security-row-intro).

## 2. What changes from your original example?

Your email policy is a good starting point. This version makes its execution order and access requirements explicit:

1. Create custom roles before trying to use them.
2. Grant them to the current instructor user before switching roles.
3. Grant warehouse use as well as database, schema, and table access.
4. Create and attach policies before testing masked access.
5. Turn off secondary roles during tests to keep the active context easy to understand.

We use `D405_` names so this exercise can coexist with your existing `DEMO_DB`, `MASKING_ADMIN`, and `MASKING_ANALYST` objects. No real username is hard-coded: `CURRENT_USER()` captures the instructor's existing login.

Like your example, the policy explicitly permits one primary role to see originals. `D405_MASKING_ADMIN` is an **unmasked-reader role in this lab**. Its name does not grant policy-administration powers. `ACCOUNTADMIN` owns and administers the lab objects; a real deployment should deliberately separate those responsibilities.

## 3. Understand the sensitive fields

**Personally Identifiable Information (PII)** identifies a person directly or through linkage.

| Field | Introduction | Analyst display in this lab |
|---|---|---|
| Customer name | Name associated with a record | Fully hidden |
| Email | Contact address | Fully hidden |
| Aadhaar | Indian identity number | Masked prefix and final four test digits |
| PAN | Permanent Account Number used for Indian tax identification | Fully hidden |
| Bank account number | Identifier for a bank account | Masked prefix and final four test digits |
| Phone | Contact number | Masked prefix and final four test digits |
| Date of birth | Personal birth date | SQL `NULL` |
| Account balance | Financial amount | SQL `NULL` |

Here **PAN means Permanent Account Number**, not the payment-card term Primary Account Number. The Income Tax Department describes PAN as a ten-character alphanumeric identifier. Our `TEST_PAN_...` values intentionally do not match a real PAN format. [PAN introduction](https://www.incometaxindia.gov.in/en/pan).

The **Unique Identification Authority of India (UIDAI)** describes masked Aadhaar as hiding the first eight digits and showing the final four. We demonstrate that display shape using invalid dummy values. This lab is not an identity-verification or legal-compliance implementation. [UIDAI masked Aadhaar](https://www.uidai.gov.in/en/283-faqs/aadhaar-online-services/e-aadhaar/1887-what-is-masked-aadhaar.html).

## 4. What each role should see

**Role-Based Access Control (RBAC)** organizes permissions through roles. The table below is our business rule, which the SQL will implement.

| Role | After column masking only | After row policy is added | Sensitive values |
|---|---|---|---|
| `D405_MASKING_ADMIN` | All four customers | All four customers | Original synthetic values |
| `D405_MASKING_ANALYST` | All four customers | South customers only | Masked or `NULL` |
| `D405_NORTH_ANALYST` | All four customers | North customers only | Masked or `NULL` |
| `D405_UNMAPPED` | All four customers | No customers | No returned rows |

`D405_UNMAPPED` demonstrates **default deny**: an approved table reader with no regional entitlement gets no rows once the row policy is installed.

An **entitlement** is an approved access allowance. We will store role-to-region entitlements in a small mapping table.

## 5. Prerequisites and collision check

Use an instructor login allowed to activate `ACCOUNTADMIN`, `USERADMIN`, and `SECURITYADMIN`. These mean Account Administrator, User and Role Administrator, and Security Administrator. Do not assign these administrator roles to learners just to run the exercise; an instructor can perform setup.

Run in one worksheet session. The warehouse is extra-small and automatically suspends after 60 seconds of inactivity; queries consume credits. The lab creates only `D405_` objects.

```sql
USE ROLE ACCOUNTADMIN;
SHOW DATABASES LIKE 'D405_DEMO_DB';
SHOW WAREHOUSES LIKE 'D405_MASKING_WH';
SHOW ROLES LIKE 'D405%';
```

Inspect any matching names. If they belong to another exercise, choose another prefix consistently. Setup uses `CREATE`, not `CREATE OR REPLACE`, so it does not silently overwrite existing objects or detach their policies.

## 6. Create roles, then assign them

The order matters: create, grant, then use. `IDENTIFIER()` treats the saved current username as an object identifier, so the script works without substituting `MAILTOGOPS`.

```sql
USE ROLE USERADMIN;

CREATE ROLE D405_MASKING_ADMIN
  COMMENT = 'D405 unmasked synthetic-data reader; not the policy owner';
CREATE ROLE D405_MASKING_ANALYST
  COMMENT = 'D405 South-region masked analyst';
CREATE ROLE D405_NORTH_ANALYST
  COMMENT = 'D405 North-region masked analyst';
CREATE ROLE D405_UNMAPPED
  COMMENT = 'D405 no-region entitlement test';

USE ROLE SECURITYADMIN;
SET D405_LAB_USER = CURRENT_USER();

GRANT ROLE D405_MASKING_ADMIN TO USER IDENTIFIER($D405_LAB_USER);
GRANT ROLE D405_MASKING_ANALYST TO USER IDENTIFIER($D405_LAB_USER);
GRANT ROLE D405_NORTH_ANALYST TO USER IDENTIFIER($D405_LAB_USER);
GRANT ROLE D405_UNMAPPED TO USER IDENTIFIER($D405_LAB_USER);

SHOW GRANTS TO USER IDENTIFIER($D405_LAB_USER);
```

We do not grant the unmasked role to an analyst role. That would give analysts a path to broader authority. If the worksheet session resets, rerun the `SET` statement before any later use of its variable.

## 7. Create a warehouse, database, and schemas

A **warehouse** supplies compute. A **schema** groups database objects. We keep business data in `SALES` and policies, tags, and entitlements in `GOVERNANCE`.

```sql
USE ROLE ACCOUNTADMIN;
USE SECONDARY ROLES NONE;

CREATE WAREHOUSE D405_MASKING_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

CREATE DATABASE D405_DEMO_DB
  COMMENT = 'D405 synthetic column and row protection lab';
CREATE SCHEMA D405_DEMO_DB.SALES
  COMMENT = 'Synthetic customer records';
CREATE SCHEMA D405_DEMO_DB.GOVERNANCE
  COMMENT = 'Policies, metadata definitions, and regional entitlements';

USE WAREHOUSE D405_MASKING_WH;
```

The privileged setup is confined to training objects. Later data-access tests run as the custom roles.

## 8. Create and populate the customer table

Identifiers are strings, not quantities. Strings preserve leading zeros and support masking without pretending that account numbers are arithmetic values. `DATE_OF_BIRTH` stays a date; `ACCOUNT_BALANCE` stays numeric.

```sql
CREATE TABLE D405_DEMO_DB.SALES.CUSTOMER (
  CUSTOMER_ID NUMBER COMMENT 'Synthetic row reference; linkable in real systems',
  CUSTOMER_NAME STRING COMMENT 'Fictional customer name',
  EMAIL STRING COMMENT 'Synthetic address in reserved example.invalid domain',
  AADHAAR_NUMBER STRING COMMENT 'Invalid dummy Aadhaar-shaped string; not an issued identifier',
  PAN_NUMBER STRING COMMENT 'Invalid TEST_PAN placeholder; Indian tax identifier concept',
  BANK_ACCOUNT_NUMBER STRING COMMENT 'Invalid TEST_BANK placeholder; not a real bank account',
  PHONE_NUMBER STRING COMMENT 'Invalid TEST_PHONE placeholder',
  DATE_OF_BIRTH DATE COMMENT 'Fictional birth date',
  REGION STRING COMMENT 'Region used by the row access policy',
  ACCOUNT_BALANCE NUMBER(12,2) COMMENT 'Fictional amount in Indian rupees'
)
COMMENT = 'One row per fictional customer; training data only';

INSERT INTO D405_DEMO_DB.SALES.CUSTOMER VALUES
  (1, 'Ravi', 'ravi@example.invalid', '000000001001', 'TEST_PAN_01',
   'TEST_BANK_00001111', 'TEST_PHONE_9001', '1990-01-15', 'SOUTH', 12500.00),
  (2, 'Priya', 'priya@example.invalid', '000000001002', 'TEST_PAN_02',
   'TEST_BANK_00002222', 'TEST_PHONE_9002', '1994-06-20', 'SOUTH', 24800.00),
  (3, 'John', 'john@example.invalid', '000000001003', 'TEST_PAN_03',
   'TEST_BANK_00003333', 'TEST_PHONE_9003', '1988-11-05', 'NORTH', 7600.00),
  (4, 'Meera', 'meera@example.invalid', '000000001004', 'TEST_PAN_04',
   NULL, 'TEST_PHONE_9004', NULL, 'WEST', NULL);

SELECT * FROM D405_DEMO_DB.SALES.CUSTOMER ORDER BY CUSTOMER_ID;
```

Expected: four rows, currently unprotected, visible to the setup role. Meera's missing values let us verify that masking preserves `NULL` rather than inventing data.

## 9. Grant the complete read path

Each reader needs warehouse, database, and schema use permissions plus table `SELECT`. Creating a role or setting its name in a policy is not an access grant.

```sql
USE ROLE SECURITYADMIN;

GRANT USAGE ON WAREHOUSE D405_MASKING_WH TO ROLE D405_MASKING_ADMIN;
GRANT USAGE ON DATABASE D405_DEMO_DB TO ROLE D405_MASKING_ADMIN;
GRANT USAGE ON SCHEMA D405_DEMO_DB.SALES TO ROLE D405_MASKING_ADMIN;
GRANT SELECT ON TABLE D405_DEMO_DB.SALES.CUSTOMER TO ROLE D405_MASKING_ADMIN;

GRANT USAGE ON WAREHOUSE D405_MASKING_WH TO ROLE D405_MASKING_ANALYST;
GRANT USAGE ON DATABASE D405_DEMO_DB TO ROLE D405_MASKING_ANALYST;
GRANT USAGE ON SCHEMA D405_DEMO_DB.SALES TO ROLE D405_MASKING_ANALYST;
GRANT SELECT ON TABLE D405_DEMO_DB.SALES.CUSTOMER TO ROLE D405_MASKING_ANALYST;

GRANT USAGE ON WAREHOUSE D405_MASKING_WH TO ROLE D405_NORTH_ANALYST;
GRANT USAGE ON DATABASE D405_DEMO_DB TO ROLE D405_NORTH_ANALYST;
GRANT USAGE ON SCHEMA D405_DEMO_DB.SALES TO ROLE D405_NORTH_ANALYST;
GRANT SELECT ON TABLE D405_DEMO_DB.SALES.CUSTOMER TO ROLE D405_NORTH_ANALYST;

GRANT USAGE ON WAREHOUSE D405_MASKING_WH TO ROLE D405_UNMAPPED;
GRANT USAGE ON DATABASE D405_DEMO_DB TO ROLE D405_UNMAPPED;
GRANT USAGE ON SCHEMA D405_DEMO_DB.SALES TO ROLE D405_UNMAPPED;
GRANT SELECT ON TABLE D405_DEMO_DB.SALES.CUSTOMER TO ROLE D405_UNMAPPED;
```

No analyst gets write access, policy ownership, or access to the entitlement table. At this intermediate point, tags and protections are not yet complete. A production release should wait until protection is configured and verified.

## 10. Tag the dataset with its purpose and ownership

A **tag** is structured metadata. A **comment** is a free-text explanation. Neither enforces masking unless explicitly connected to a policy.

These user-defined labels describe the table. They are not Snowflake built-in classifications.

```sql
USE ROLE ACCOUNTADMIN;

CREATE TAG D405_DEMO_DB.GOVERNANCE.DATA_OWNER
  ALLOWED_VALUES 'CUSTOMER_OPERATIONS'
  COMMENT = 'Business accountability; not technical object ownership';
CREATE TAG D405_DEMO_DB.GOVERNANCE.CONTAINS_PII
  ALLOWED_VALUES 'YES', 'NO';
CREATE TAG D405_DEMO_DB.GOVERNANCE.PURPOSE
  ALLOWED_VALUES 'MASKING_TRAINING';
CREATE TAG D405_DEMO_DB.GOVERNANCE.SENSITIVITY
  ALLOWED_VALUES 'INTERNAL', 'RESTRICTED';
CREATE TAG D405_DEMO_DB.GOVERNANCE.DATA_CATEGORY
  ALLOWED_VALUES 'CUSTOMER_REFERENCE', 'PERSON_NAME', 'EMAIL', 'AADHAAR',
    'INDIAN_TAX_PAN', 'BANK_ACCOUNT', 'PHONE', 'BIRTH_DATE', 'SALES_REGION',
    'FINANCIAL_BALANCE';

ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER SET TAG
  D405_DEMO_DB.GOVERNANCE.DATA_OWNER = 'CUSTOMER_OPERATIONS',
  D405_DEMO_DB.GOVERNANCE.CONTAINS_PII = 'YES',
  D405_DEMO_DB.GOVERNANCE.PURPOSE = 'MASKING_TRAINING';
```

`CONTAINS_PII = YES` models the classification of a real customer dataset; the actual rows remain synthetic. Do not store customer secrets or identifier values in metadata.

## 11. Tag each column with precise meaning

```sql
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN CUSTOMER_ID SET TAG
  D405_DEMO_DB.GOVERNANCE.DATA_CATEGORY = 'CUSTOMER_REFERENCE',
  D405_DEMO_DB.GOVERNANCE.SENSITIVITY = 'RESTRICTED';
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN CUSTOMER_NAME SET TAG
  D405_DEMO_DB.GOVERNANCE.DATA_CATEGORY = 'PERSON_NAME',
  D405_DEMO_DB.GOVERNANCE.SENSITIVITY = 'RESTRICTED';
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN EMAIL SET TAG
  D405_DEMO_DB.GOVERNANCE.DATA_CATEGORY = 'EMAIL',
  D405_DEMO_DB.GOVERNANCE.SENSITIVITY = 'RESTRICTED';
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN AADHAAR_NUMBER SET TAG
  D405_DEMO_DB.GOVERNANCE.DATA_CATEGORY = 'AADHAAR',
  D405_DEMO_DB.GOVERNANCE.SENSITIVITY = 'RESTRICTED';
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN PAN_NUMBER SET TAG
  D405_DEMO_DB.GOVERNANCE.DATA_CATEGORY = 'INDIAN_TAX_PAN',
  D405_DEMO_DB.GOVERNANCE.SENSITIVITY = 'RESTRICTED';
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN BANK_ACCOUNT_NUMBER SET TAG
  D405_DEMO_DB.GOVERNANCE.DATA_CATEGORY = 'BANK_ACCOUNT',
  D405_DEMO_DB.GOVERNANCE.SENSITIVITY = 'RESTRICTED';
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN PHONE_NUMBER SET TAG
  D405_DEMO_DB.GOVERNANCE.DATA_CATEGORY = 'PHONE',
  D405_DEMO_DB.GOVERNANCE.SENSITIVITY = 'RESTRICTED';
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN DATE_OF_BIRTH SET TAG
  D405_DEMO_DB.GOVERNANCE.DATA_CATEGORY = 'BIRTH_DATE',
  D405_DEMO_DB.GOVERNANCE.SENSITIVITY = 'RESTRICTED';
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN REGION SET TAG
  D405_DEMO_DB.GOVERNANCE.DATA_CATEGORY = 'SALES_REGION',
  D405_DEMO_DB.GOVERNANCE.SENSITIVITY = 'INTERNAL';
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN ACCOUNT_BALANCE SET TAG
  D405_DEMO_DB.GOVERNANCE.DATA_CATEGORY = 'FINANCIAL_BALANCE',
  D405_DEMO_DB.GOVERNANCE.SENSITIVITY = 'RESTRICTED';

SELECT * FROM TABLE(
  D405_DEMO_DB.INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
    'D405_DEMO_DB.SALES.CUSTOMER', 'TABLE'
  )
);
```

The customer reference stays visible for lab comparisons even though it is linkable and classified as restricted. A sensitivity label does not mean every field must use exactly the same protection. The approved use determines handling.

The inspection function includes inherited associations, so table labels can appear for columns too. [Column tag inspection](https://docs.snowflake.com/en/sql-reference/functions/tag_references_all_columns).

## 12. Create the column masking policies

A **dynamic masking policy** changes the value returned during a query while leaving the stored original intact. The input and output types must match. Dates and numbers cannot simply return the text `********`, so their policies return typed `NULL` instead. [CREATE MASKING POLICY](https://docs.snowflake.com/en/sql-reference/sql/create-masking-policy).

```sql
USE ROLE ACCOUNTADMIN;

CREATE MASKING POLICY D405_DEMO_DB.GOVERNANCE.FULL_TEXT_MASK
AS (VAL STRING) RETURNS STRING ->
  CASE
    WHEN VAL IS NULL THEN NULL
    WHEN CURRENT_ROLE() = 'D405_MASKING_ADMIN' THEN VAL
    ELSE '********'
  END;

CREATE MASKING POLICY D405_DEMO_DB.GOVERNANCE.AADHAAR_MASK
AS (VAL STRING) RETURNS STRING ->
  CASE
    WHEN VAL IS NULL THEN NULL
    WHEN CURRENT_ROLE() = 'D405_MASKING_ADMIN' THEN VAL
    ELSE 'XXXX-XXXX-' || RIGHT(VAL, 4)
  END;

CREATE MASKING POLICY D405_DEMO_DB.GOVERNANCE.LAST_FOUR_MASK
AS (VAL STRING) RETURNS STRING ->
  CASE
    WHEN VAL IS NULL THEN NULL
    WHEN CURRENT_ROLE() = 'D405_MASKING_ADMIN' THEN VAL
    ELSE '********' || RIGHT(VAL, 4)
  END;

CREATE MASKING POLICY D405_DEMO_DB.GOVERNANCE.BIRTH_DATE_MASK
AS (VAL DATE) RETURNS DATE ->
  CASE WHEN CURRENT_ROLE() = 'D405_MASKING_ADMIN'
       THEN VAL ELSE NULL END;

CREATE MASKING POLICY D405_DEMO_DB.GOVERNANCE.BALANCE_MASK
AS (VAL NUMBER(12,2)) RETURNS NUMBER(12,2) ->
  CASE WHEN CURRENT_ROLE() = 'D405_MASKING_ADMIN'
       THEN VAL ELSE NULL END;
```

`RIGHT(VAL, 4)` takes the final four characters; `||` joins strings. Partial display is a chosen disclosure, not anonymity. This demonstration assumes normalized test inputs and does not validate identity-number formats.

## 13. Attach policies to eight sensitive columns

Creating a policy does not attach it. The following statements connect each field to its protection.

```sql
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN CUSTOMER_NAME
  SET MASKING POLICY D405_DEMO_DB.GOVERNANCE.FULL_TEXT_MASK;
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN EMAIL
  SET MASKING POLICY D405_DEMO_DB.GOVERNANCE.FULL_TEXT_MASK;
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN PAN_NUMBER
  SET MASKING POLICY D405_DEMO_DB.GOVERNANCE.FULL_TEXT_MASK;
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN AADHAAR_NUMBER
  SET MASKING POLICY D405_DEMO_DB.GOVERNANCE.AADHAAR_MASK;
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN BANK_ACCOUNT_NUMBER
  SET MASKING POLICY D405_DEMO_DB.GOVERNANCE.LAST_FOUR_MASK;
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN PHONE_NUMBER
  SET MASKING POLICY D405_DEMO_DB.GOVERNANCE.LAST_FOUR_MASK;
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN DATE_OF_BIRTH
  SET MASKING POLICY D405_DEMO_DB.GOVERNANCE.BIRTH_DATE_MASK;
ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN ACCOUNT_BALANCE
  SET MASKING POLICY D405_DEMO_DB.GOVERNANCE.BALANCE_MASK;
```

Only `CUSTOMER_ID` and `REGION` are left without column masks. Region will be the input to the row policy, keeping that decision separate from masked fields. [ALTER TABLE protection operations](https://docs.snowflake.com/en/sql-reference/sql/alter-table).

## 14. Test column masking before adding row filtering

```sql
USE ROLE D405_MASKING_ANALYST;
USE SECONDARY ROLES NONE;
USE WAREHOUSE D405_MASKING_WH;

SELECT CURRENT_USER(), CURRENT_ROLE();
SELECT * FROM D405_DEMO_DB.SALES.CUSTOMER ORDER BY CUSTOMER_ID;
SELECT COUNT(*) AS VISIBLE_ROWS FROM D405_DEMO_DB.SALES.CUSTOMER;
```

Expected count: **4**. Masking has changed values, not row visibility.

For customer 1, selected results are:

| Field | Analyst result |
|---|---|
| Customer name | `********` |
| Email | `********` |
| Aadhaar | `XXXX-XXXX-1001` |
| PAN | `********` |
| Bank account | `********1111` |
| Phone | `********9001` |
| Birth date | `NULL` |
| Balance | `NULL` |

Customer 4's absent bank account remains `NULL`. Do not substitute a fake balance of zero: zero is a real financial value and could mislead calculations.

## 15. Test the unmasked role and understand CURRENT_ROLE

```sql
USE ROLE D405_MASKING_ADMIN;
USE SECONDARY ROLES NONE;
USE WAREHOUSE D405_MASKING_WH;

SELECT CURRENT_USER(), CURRENT_ROLE();
SELECT * FROM D405_DEMO_DB.SALES.CUSTOMER ORDER BY CUSTOMER_ID;
```

Expected: all four rows with original synthetic values. The username is unchanged; only the active primary role changed.

`CURRENT_ROLE()` reports the primary account role. Our exact comparison deliberately does not mean “any role inheriting from this role” or “any administrator.” Even `ACCOUNTADMIN` does not match the string `D405_MASKING_ADMIN`. [CURRENT_ROLE](https://docs.snowflake.com/en/sql-reference/functions/current_role).

An inheritance-aware design could instead use an appropriate role-in-session function, but that changes who qualifies. Keep the business rule and tests aligned. Disabling secondary roles makes the demonstration easier to interpret; it is not a substitute for reviewing all grants.

## 16. Create a regional entitlement table

This mapping says which analyst role is allowed to see which region. Adding a row grants a regional allowance through the policy; removing it removes that allowance, provided there is no other matching entry or bypass.

```sql
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE D405_MASKING_WH;

CREATE TABLE D405_DEMO_DB.GOVERNANCE.ROLE_REGION_ACCESS (
  ROLE_NAME STRING,
  ALLOWED_REGION STRING
)
COMMENT = 'Policy lookup: approved primary-role to region allowances';

INSERT INTO D405_DEMO_DB.GOVERNANCE.ROLE_REGION_ACCESS VALUES
  ('D405_MASKING_ANALYST', 'SOUTH'),
  ('D405_NORTH_ANALYST', 'NORTH');

SELECT * FROM D405_DEMO_DB.GOVERNANCE.ROLE_REGION_ACCESS
ORDER BY ROLE_NAME;
```

There is no entry for `D405_UNMAPPED`. The unmasked role will have an explicit all-region allowance in the policy body. Analysts do not receive direct access to this governance table.

## 17. Create and attach the row access policy

The policy returns true when a row is allowed. It permits the unmasked role, or a role/region match in the entitlement table. A missing match produces no visible row.

```sql
USE ROLE ACCOUNTADMIN;

CREATE ROW ACCESS POLICY D405_DEMO_DB.GOVERNANCE.CUSTOMER_REGION_ACCESS
AS (ROW_REGION STRING) RETURNS BOOLEAN ->
  CURRENT_ROLE() = 'D405_MASKING_ADMIN'
  OR EXISTS (
    SELECT 1
    FROM D405_DEMO_DB.GOVERNANCE.ROLE_REGION_ACCESS M
    WHERE M.ROLE_NAME = CURRENT_ROLE()
      AND M.ALLOWED_REGION = ROW_REGION
  );

ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER
  ADD ROW ACCESS POLICY D405_DEMO_DB.GOVERNANCE.CUSTOMER_REGION_ACCESS
  ON (REGION);
```

Snowflake evaluates the mapping lookup using the policy owner's privileges, so consumers do not need to read the mapping directly. The context comparison still uses the querying session's primary role. Here the policy owner also owns the lookup table. [CREATE ROW ACCESS POLICY](https://docs.snowflake.com/en/sql-reference/sql/create-row-access-policy); [row-policy evaluation](https://docs.snowflake.com/en/user-guide/security-row-intro).

When row and masking policies protect the same table, the row policy is evaluated first. Eligible rows then receive the permitted column representations. The SQL below lets you observe both effects.

## 18. South analyst: two rows with masked fields

```sql
USE ROLE D405_MASKING_ANALYST;
USE SECONDARY ROLES NONE;
USE WAREHOUSE D405_MASKING_WH;

SELECT CURRENT_USER(), CURRENT_ROLE();
SELECT * FROM D405_DEMO_DB.SALES.CUSTOMER ORDER BY CUSTOMER_ID;
SELECT COUNT(*) AS VISIBLE_ROWS FROM D405_DEMO_DB.SALES.CUSTOMER;
```

Expected count: **2**.

| Customer reference | Region | Email | Aadhaar | Bank account |
|---|---|---|---|---|
| 1 | SOUTH | `********` | `XXXX-XXXX-1001` | `********1111` |
| 2 | SOUTH | `********` | `XXXX-XXXX-1002` | `********2222` |

John and Meera are absent from the result. They are not replaced with blank rows and have not been deleted.

## 19. North analyst and a role with no entitlement

```sql
USE ROLE D405_NORTH_ANALYST;
USE SECONDARY ROLES NONE;
USE WAREHOUSE D405_MASKING_WH;

SELECT * FROM D405_DEMO_DB.SALES.CUSTOMER ORDER BY CUSTOMER_ID;
SELECT COUNT(*) AS VISIBLE_ROWS FROM D405_DEMO_DB.SALES.CUSTOMER;

USE ROLE D405_UNMAPPED;
USE SECONDARY ROLES NONE;

SELECT * FROM D405_DEMO_DB.SALES.CUSTOMER ORDER BY CUSTOMER_ID;
SELECT COUNT(*) AS VISIBLE_ROWS FROM D405_DEMO_DB.SALES.CUSTOMER;
```

Expected: North sees **one** masked record, customer 3. Unmapped sees **zero** records.

Zero records is a successful policy-filtered query, not a missing-privilege error. A role lacking table `SELECT` would instead be denied the table operation.

## 20. Unmasked role: four original records

```sql
USE ROLE D405_MASKING_ADMIN;
USE SECONDARY ROLES NONE;
USE WAREHOUSE D405_MASKING_WH;

SELECT * FROM D405_DEMO_DB.SALES.CUSTOMER ORDER BY CUSTOMER_ID;
SELECT COUNT(*) AS VISIBLE_ROWS FROM D405_DEMO_DB.SALES.CUSTOMER;
```

Expected: **4**, with original synthetic fields. This proves that analysts' hidden rows still exist.

If you query this protected table with primary role `ACCOUNTADMIN`, this particular row policy returns no rows because that role has neither a mapping nor the named exception. Ownership does not silently change the policy's condition. Administrators who can alter or remove protection still have powerful capabilities, so monitor policy administration separately.

## 21. Change access by updating the mapping

Temporarily give the South analyst access to North as well:

```sql
USE ROLE ACCOUNTADMIN;
INSERT INTO D405_DEMO_DB.GOVERNANCE.ROLE_REGION_ACCESS
VALUES ('D405_MASKING_ANALYST', 'NORTH');

USE ROLE D405_MASKING_ANALYST;
USE SECONDARY ROLES NONE;
SELECT CUSTOMER_ID, REGION, EMAIL
FROM D405_DEMO_DB.SALES.CUSTOMER ORDER BY CUSTOMER_ID;
```

Expected: customers **1, 2, and 3**, all with masked email. Increasing row access does not remove column masking.

Restore the original entitlement:

```sql
USE ROLE ACCOUNTADMIN;
DELETE FROM D405_DEMO_DB.GOVERNANCE.ROLE_REGION_ACCESS
WHERE ROLE_NAME = 'D405_MASKING_ANALYST'
  AND ALLOWED_REGION = 'NORTH';

USE ROLE D405_MASKING_ANALYST;
USE SECONDARY ROLES NONE;
SELECT COUNT(*) AS VISIBLE_ROWS FROM D405_DEMO_DB.SALES.CUSTOMER;
```

Expected: **2** again. In a real system, approve, audit, and expire entitlement changes instead of treating this mapping as an ordinary editable business table.

## 22. Verify that analysts cannot edit entitlements

Run this block separately: the error is intentional.

```sql
USE ROLE D405_MASKING_ANALYST;
USE SECONDARY ROLES NONE;

-- EXPECTED ERROR: no governance schema/table read privileges.
SELECT * FROM D405_DEMO_DB.GOVERNANCE.ROLE_REGION_ACCESS;
```

Expected: an authorization error, which may say the object does not exist or is not authorized. The role has also received no write permissions on this mapping.

The customer query can still succeed because the row policy uses its owner's privileges for the lookup. This is useful separation: analysts consume approved access without administering it.

## 23. Inspect actual policies and tag coverage

```sql
USE ROLE ACCOUNTADMIN;

SHOW MASKING POLICIES IN SCHEMA D405_DEMO_DB.GOVERNANCE;
SHOW ROW ACCESS POLICIES IN SCHEMA D405_DEMO_DB.GOVERNANCE;
DESCRIBE MASKING POLICY D405_DEMO_DB.GOVERNANCE.AADHAAR_MASK;
DESCRIBE ROW ACCESS POLICY D405_DEMO_DB.GOVERNANCE.CUSTOMER_REGION_ACCESS;

SELECT * FROM TABLE(
  D405_DEMO_DB.INFORMATION_SCHEMA.POLICY_REFERENCES(
    REF_ENTITY_NAME => 'D405_DEMO_DB.SALES.CUSTOMER',
    REF_ENTITY_DOMAIN => 'TABLE'
  )
);

SELECT * FROM TABLE(
  D405_DEMO_DB.INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
    'D405_DEMO_DB.SALES.CUSTOMER', 'TABLE'
  )
);
```

Before the optional extension, expect five masking definitions applied across eight columns, plus one row policy attached on `REGION`. Inspect the returned associations rather than just counting policy definitions. [POLICY_REFERENCES](https://docs.snowflake.com/en/sql-reference/functions/policy_references).

Tags answer “what kind of data is this?” Policy associations answer “what is connected to protect it?” Role-by-role queries answer “what does the consumer actually receive?” All three checks matter.

## 24. Optional: make the email tag enforce masking

The earlier tags are descriptive. This extension connects a dedicated tag to the existing text policy. It illustrates **tag-based masking**, which can scale protection to compatible tagged columns. A directly attached column policy takes precedence over a tag-based policy. [Tag-based masking policies](https://docs.snowflake.com/en/user-guide/tag-based-masking-policies).

Run only after the core lab. We establish the tag-based protection before removing the direct email attachment.

```sql
USE ROLE ACCOUNTADMIN;

CREATE TAG D405_DEMO_DB.GOVERNANCE.EMAIL_PROTECTION
  ALLOWED_VALUES 'MASK_REQUIRED'
  COMMENT = 'Enforcement tag: compatible columns use FULL_TEXT_MASK';

ALTER TAG D405_DEMO_DB.GOVERNANCE.EMAIL_PROTECTION
  SET MASKING POLICY D405_DEMO_DB.GOVERNANCE.FULL_TEXT_MASK;

ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN EMAIL
  SET TAG D405_DEMO_DB.GOVERNANCE.EMAIL_PROTECTION = 'MASK_REQUIRED';

ALTER TABLE D405_DEMO_DB.SALES.CUSTOMER MODIFY COLUMN EMAIL
  UNSET MASKING POLICY;

USE ROLE D405_MASKING_ANALYST;
USE SECONDARY ROLES NONE;
SELECT CUSTOMER_ID, EMAIL FROM D405_DEMO_DB.SALES.CUSTOMER ORDER BY CUSTOMER_ID;

USE ROLE D405_MASKING_ADMIN;
USE SECONDARY ROLES NONE;
SELECT CUSTOMER_ID, EMAIL FROM D405_DEMO_DB.SALES.CUSTOMER ORDER BY CUSTOMER_ID;
```

Expected: South still sees two masked emails; the unmasked role sees four original synthetic emails. The email policy now comes through the tag. Other columns retain direct policy attachments.

Do not attach one broad text-masking tag to every column just because it is sensitive: type compatibility and the desired representation can differ.

## 25. What these controls do not mean

- Last-four display reveals information intentionally; it is not anonymity.
- A masked birth date is not erased from storage.
- A customer reference can still link records to a person in another dataset.
- A permitted export becomes another copy that needs its own controls.
- A row policy is not a complete write-validation rule. This lab grants only `SELECT` to consumers.
- Returning `NULL` affects calculations: consumers must not interpret a masked balance as zero or a complete total.
- Role names are not legal justification to collect identity or banking data.

In a real solution, define collection purpose, access approvals, retention, and incident handling with the responsible teams. This lesson demonstrates platform behavior using synthetic records.

## 26. Troubleshooting checklist

| Observation | Check |
|---|---|
| Cannot use the masking role | Was it created and assigned before `USE ROLE`? |
| No warehouse selected or authorized | Select `D405_MASKING_WH` and verify its `USAGE` grant |
| Table cannot be read | Verify database, schema, and table privileges |
| Policy feature unavailable | Confirm Enterprise Edition or higher |
| Analyst sees all four rows | Was the row policy attached yet? |
| Analyst sees originals | Check exact primary role, policy associations, and any later changes |
| Administrator sees zero rows | The exact-match row rule only exempts `D405_MASKING_ADMIN` |
| Type error in a mask | Match policy input/output to the protected column type |
| Missing values appear as empty cells | Inspect SQL `NULL`; this is different from an empty string |
| Rerun creates duplicate rows | Setup is a one-pass lab; clean up before a full restart |

Do not repeatedly use replacement commands to fix policy errors. Replacing protected objects can change attachments, grants, or ownership. Inspect the existing state first.

## 27. Cleanup

Run after finishing the core lab and optional extension. Verify these names still belong only to this exercise. The database contains all policies, tags, mappings, and synthetic data; no policy references objects outside it.

```sql
USE ROLE ACCOUNTADMIN;
USE SECONDARY ROLES NONE;

DROP DATABASE IF EXISTS D405_DEMO_DB;
DROP WAREHOUSE IF EXISTS D405_MASKING_WH;

USE ROLE USERADMIN;
DROP ROLE IF EXISTS D405_MASKING_ADMIN;
DROP ROLE IF EXISTS D405_MASKING_ANALYST;
DROP ROLE IF EXISTS D405_NORTH_ANALYST;
DROP ROLE IF EXISTS D405_UNMAPPED;

SHOW ROLES LIKE 'D405%';
```

Dropping the roles also removes their lab assignments. Database removal follows Snowflake's recovery-retention behavior; it is not a promise of immediate physical erasure of every historical version.

## 28. Quick review

1. **What does a masking policy change?** The returned representation of protected column values.
2. **What does a row access policy change?** Which records are returned.
3. **Why can the South analyst see two rows but no full emails?** Both policies apply together.
4. **Why use strings for identifiers?** They preserve representation and leading zeros; identifiers are not quantities.
5. **Why do dates and balances return `NULL`?** Their policies must retain compatible data types.
6. **Why does the mapping table stay private?** Consumers do not need to administer their own entitlements.
7. **Does a PII tag automatically mask data?** No; the optional enforcement tag explicitly links to a policy.
8. **Does `ACCOUNTADMIN` automatically satisfy an exact role-name condition?** No; administrative power and a policy condition are separate.

| Abbreviation | Expansion |
|---|---|
| SQL | Structured Query Language |
| PII | Personally Identifiable Information |
| PAN | Permanent Account Number in this notebook |
| UIDAI | Unique Identification Authority of India |
| RBAC | Role-Based Access Control |
| ID | Identifier, as in customer identifier |
| DB | Database, used in object names |
| WH | Warehouse, used in object names |

Official references are linked beside the corresponding explanations and commands. The lab's role names, tagging vocabulary, and access rules are illustrative organizational choices.